# 00b — Stitch images acquired with different beamstop positions

All images are aligned and intensity-normalized to one explicitly selected reference image per polarization. The fixed detector mask never moves. Each beamstop mask may be shifted independently. Their union is the `mask_pixel` used for registration, normalization, and stitching.

The reference image remains unchanged wherever it is valid. Other images only fill its masked region; multiple valid fill values are averaged.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

BASEFOLDER = Path.cwd().resolve()
sys.path.insert(0, str(BASEFOLDER / "library"))

from beamstop_stitching import shift_mask, stitch_images
from data_loading import Frame, SextantsNexusLoader, load_average
from image_preprocessing import fit_dark_frame, fit_horizontal_band, load_detector_masks

%matplotlib widget
print("Base folder:", BASEFOLDER)


## Select the input images

In [ ]:
# One position in these parallel lists defines one input to stitching.
# Every IMAGE_ID_GROUPS item may be one ID or a list of IDs to average first.
INPUT_KIND = "raw"  # "raw" or "preprocessed"
IMAGE_ID_GROUPS = [
    452,
    453,
    464,
    465,
    466,
    467,
    468,
    469,
    470,
    471,
]
POLARIZATIONS = ["+", "-", "+", "-", "+", "-", "+", "-", "+", "-"]

# Each raw input may use a different dark. A dark entry may also be a list.
# These entries are ignored when INPUT_KIND="preprocessed" because notebook 01a
# has already fitted and subtracted the dark from those files.
DARK_ID_GROUPS = [443, 443, 443, 443, 443, 443, 443, 443, 443, 443]

# The fixed detector mask is always mask_detector.png. Each raw input selects
# one mask_beamstop_<ID>.png, which can then be moved by the corresponding shift.
BEAMSTOP_MASK_IDS = [95, 95, 95, 95, 95, 95, 95, 95, 95, 95]
MASK_SHIFTS = [
    (0, 0), (0, 0),
    (52, -5), (52, -5),
    (-52, -5), (-52, -5),
    (0, -20), (0, -20),
    (-10, 5), (-10, 5),
]

RAW_FOLDER = Path("/nfs/ruche/sextants-soleil/com-sextants/COMET_20260902_Cocoons_Laser/")
PREPROCESSED_FOLDER = BASEFOLDER / "processed" / "cleaned_acquisitions"
MASK_FOLDER = BASEFOLDER / "processed" / "mask_pixels"
USER = "rb"

input_count = len(IMAGE_ID_GROUPS)
if not all(len(values) == input_count for values in (
    POLARIZATIONS, DARK_ID_GROUPS, BEAMSTOP_MASK_IDS, MASK_SHIFTS
)):
    raise ValueError("All parallel input lists must have the same length")
print("Fixed detector mask:", MASK_FOLDER / "mask_detector.png")
for index, values in enumerate(zip(
    IMAGE_ID_GROUPS, POLARIZATIONS, DARK_ID_GROUPS, BEAMSTOP_MASK_IDS, MASK_SHIFTS
)):
    image_ids, polarization, dark_ids, beamstop_id, shift = values
    print(f"input {index}: {polarization}, images={image_ids}, darks={dark_ids}, "
          f"mask_beamstop_{beamstop_id}.png, shift={shift}")


## Load and average every input group


In [ ]:
# Dark fitting always uses this corner and excludes the fixed detector mask.
DARK_FIT_ROWS = slice(0, 600)
DARK_FIT_COLUMNS = slice(0, 200)
DARK_FIT_PERCENTILE = 100
DARK_FIT_STRIDE = 1


def as_id_list(image_ids):
    if isinstance(image_ids, (list, tuple, np.ndarray)):
        return [int(image_id) for image_id in image_ids]
    return [int(image_ids)]


def average_preprocessed_files(image_ids, beamstop_shift):
    images = []
    detector_masks = []
    beamstop_masks = []
    energies = []
    sources = []
    for image_id in as_id_list(image_ids):
        input_file = PREPROCESSED_FOLDER / f"cleaned_ImId_{image_id:04d}_{USER}.npz"
        with np.load(input_file, allow_pickle=False) as saved:
            images.append(np.asarray(saved["image"], dtype=float))
            detector_masks.append(np.asarray(saved["mask_detector"], dtype=np.uint8))
            beamstop_masks.append(shift_mask(
                np.asarray(saved["mask_beamstop"], dtype=np.uint8), beamstop_shift
            ))
            energies.append(float(saved["energy_eV"]))
        sources.append(input_file)

    image_stack = np.stack(images)
    detector_mask = np.maximum.reduce(detector_masks).astype(np.uint8)
    beamstop_stack = np.stack(beamstop_masks).astype(np.uint8)
    pixel_mask_stack = np.clip(detector_mask[None, ...] + beamstop_stack, 0, 1)
    valid = (pixel_mask_stack == 0) & np.isfinite(image_stack)
    counts = valid.sum(axis=0)
    averaged = np.divide(
        np.where(valid, image_stack, 0).sum(axis=0),
        counts,
        out=np.zeros(image_stack.shape[1:], dtype=float),
        where=counts > 0,
    )
    beamstop_mask = (
        (counts == 0) & (detector_mask == 0)
    ).astype(np.uint8)
    pixel_mask = np.clip(detector_mask + beamstop_mask, 0, 1).astype(np.uint8)
    return averaged, detector_mask, beamstop_mask, pixel_mask, float(np.mean(energies)), sources


loader = SextantsNexusLoader(RAW_FOLDER)
frames = []
mask_detectors = []
mask_beamstops = []
mask_pixels = []
group_names = []
group_id_lists = []
energies_eV = []
dark_fit_plots = []

for input_index, values in enumerate(zip(
    IMAGE_ID_GROUPS, DARK_ID_GROUPS, BEAMSTOP_MASK_IDS, MASK_SHIFTS
)):
    image_ids, dark_ids, beamstop_id, beamstop_shift = values
    image_id_list = as_id_list(image_ids)
    group_name = "+".join(str(image_id) for image_id in image_id_list)

    if INPUT_KIND == "raw":
        # load_average accepts one ID or a list and always returns a float average.
        loaded = load_average(loader, image_ids)
        raw_average = np.asarray(loaded.image, dtype=float)
        dark_average = np.asarray(load_average(loader, dark_ids).image, dtype=float)
        mask_detector, mask_beamstop, _ = load_detector_masks(
            MASK_FOLDER, beamstop_id, raw_average.shape
        )
        mask_beamstop = shift_mask(mask_beamstop, beamstop_shift)
        mask_pixel = np.clip(mask_detector + mask_beamstop, 0, 1).astype(np.uint8)
        image, scale, offset, dark_values, fitted_pixels = fit_dark_frame(
            raw_average,
            dark_average,
            DARK_FIT_ROWS,
            DARK_FIT_COLUMNS,
            percentile=DARK_FIT_PERCENTILE,
            stride=DARK_FIT_STRIDE,
            mask=mask_detector,
        )
        image_values = raw_average[DARK_FIT_ROWS, DARK_FIT_COLUMNS][
            ::DARK_FIT_STRIDE, ::DARK_FIT_STRIDE
        ].ravel()
        dark_fit_plots.append((group_name, dark_values, image_values, fitted_pixels, scale, offset))
        exposure = float(loaded.exposure)
        energy_eV = float(loaded.metadata["energy_eV"])
        source_name = loaded.source
    elif INPUT_KIND == "preprocessed":
        image, mask_detector, mask_beamstop, mask_pixel, energy_eV, sources = (
            average_preprocessed_files(image_ids, beamstop_shift)
        )
        exposure = 1.0
        source_name = sources[0]
    else:
        raise ValueError('INPUT_KIND must be "raw" or "preprocessed"')

    frames.append(Frame(group_name, image, exposure, Path(source_name)))
    mask_detectors.append(mask_detector)
    mask_beamstops.append(mask_beamstop)
    mask_pixels.append(mask_pixel)
    group_names.append(group_name)
    group_id_lists.append(image_id_list)
    energies_eV.append(energy_eV)
    print(f"Loaded input {input_index}, IDs {image_id_list}: average shape {image.shape}")

energy_eV = float(np.mean(energies_eV))
print("Photon-energy range:", min(energies_eV), "to", max(energies_eV), "eV")


## Inspect fitted dark normalization and the separate masks


In [ ]:
if dark_fit_plots:
    fig, axes = plt.subplots(len(dark_fit_plots), 1, figsize=(7, 4 * len(dark_fit_plots)), squeeze=False)
    for axis, fit_data in zip(axes.flat, dark_fit_plots):
        group_name, dark_values, image_values, fitted_pixels, scale, offset = fit_data
        step = max(1, fitted_pixels.sum() // 5000)
        x = dark_values[fitted_pixels][::step]
        y = image_values[fitted_pixels][::step]
        axis.scatter(x, y, s=3, alpha=0.15, label="corner pixels")
        fit_x = np.linspace(x.min(), x.max(), 200)
        axis.plot(fit_x, scale * fit_x + offset, color="red", linewidth=2,
                  label=f"fit: {scale:.5g} x + {offset:.5g}")
        axis.set_title(f"input {group_name}: dark normalization")
        axis.set_xlabel("Dark intensity")
        axis.set_ylabel("Image intensity")
        axis.legend()
        axis.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

fig, axes = plt.subplots(len(IMAGE_ID_GROUPS), 3, figsize=(12, 3.5 * len(IMAGE_ID_GROUPS)), squeeze=False)
for row, group_name in enumerate(group_names):
    for axis, mask, title in zip(
        axes[row],
        (mask_detectors[row], mask_beamstops[row], mask_pixels[row]),
        ("fixed mask_detector", "shifted mask_beamstop", "combined mask_pixel"),
    ):
        axis.imshow(mask, cmap="gray", vmin=0, vmax=1)
        axis.set_title(f"input {group_name}: {title}")
        axis.set_axis_off()
plt.tight_layout()
plt.show()


## Optionally correct the horizontal band, then stitch to each reference


In [ ]:
# The band algorithm is always available; disable it without deleting code.
CORRECT_HORIZONTAL_BAND = False
BAND_EDGE_COLUMNS = 20
BAND_SKIPPED_EDGE_COLUMNS = 0
BAND_POLYNOMIAL_ORDER = 2
BAND_CENTER = 1024
BAND_WIDTH = 80
BAND_EDGE = 11
SUBTRACT_POLYNOMIAL_BACKGROUND = False

if CORRECT_HORIZONTAL_BAND:
    corrected_frames = []
    for frame, mask_detector, mask_pixel in zip(frames, mask_detectors, mask_pixels):
        band_mask = mask_pixel.copy()
        band_mask[:20, :] = 1
        band_mask[-200:, :] = 1
        fit = fit_horizontal_band(
            frame.image,
            edge_columns=BAND_EDGE_COLUMNS,
            skipped_edge_columns=BAND_SKIPPED_EDGE_COLUMNS,
            polynomial_order=BAND_POLYNOMIAL_ORDER,
            band_center=BAND_CENTER,
            band_width=BAND_WIDTH,
            band_edge=BAND_EDGE,
            mask=band_mask,
        )
        corrected_image = frame.image - fit.band_image
        if SUBTRACT_POLYNOMIAL_BACKGROUND:
            corrected_image -= fit.polynomial_image
        corrected_frames.append(Frame(frame.image_id, corrected_image, frame.exposure, frame.source))

        fig, axis = plt.subplots(figsize=(9, 4))
        axis.plot(fit.rows, fit.measured_profile, label="measured")
        axis.plot(fit.rows, fit.fitted_profile, linewidth=2, label="complete fit")
        axis.plot(fit.rows, fit.polynomial_profile, "--", label=f"polynomial order {BAND_POLYNOMIAL_ORDER}")
        axis.fill_between(fit.rows, fit.polynomial_profile, fit.fitted_profile, alpha=0.25, label="band")
        axis.set_title(f"ID {frame.image_id}: horizontal-band fit")
        axis.legend()
        axis.grid(alpha=0.2)
        plt.tight_layout()
        plt.show()
    frames = corrected_frames

# Select the reference rows here, immediately before stitching.
PLUS_REFERENCE_INPUT = 0
MINUS_REFERENCE_INPUT = 1
for reference_input, polarization in (
    (PLUS_REFERENCE_INPUT, "+"), (MINUS_REFERENCE_INPUT, "-")
):
    if not 0 <= reference_input < len(IMAGE_ID_GROUPS):
        raise ValueError("Reference input index is outside IMAGE_ID_GROUPS")
    if POLARIZATIONS[reference_input] != polarization:
        raise ValueError(f"Reference input {reference_input} is not polarization {polarization}")

# Registration and intensity fitting use only this central mutually valid ROI.
FIND_IMAGE_SHIFTS = True
MAX_IMAGE_SHIFT = 10.0
REGISTRATION_UPSAMPLE = 10
FIT_ORDER = 1  # 1 = factor + offset; 2 or higher = polynomial mapping.
FIT_PERCENTILES = (2, 98)
ALIGNMENT_ROI = np.s_[800:-800, 800:-800]

stitch_results = []
stitched_image_ids = []
for polarization, reference_index in (
    ("+", PLUS_REFERENCE_INPUT),
    ("-", MINUS_REFERENCE_INPUT),
):
    selected = [index for index, value in enumerate(POLARIZATIONS) if value == polarization]
    if reference_index not in selected:
        raise ValueError(f"Reference input {reference_index} is not a {polarization} image")
    selected.remove(reference_index)
    selected.insert(0, reference_index)

    selected_frames = [frames[index] for index in selected]
    selected_masks = [mask_pixels[index] for index in selected]
    result = stitch_images(
        selected_frames,
        selected_masks,
        register=FIND_IMAGE_SHIFTS,
        max_shift=MAX_IMAGE_SHIFT,
        upsample_factor=REGISTRATION_UPSAMPLE,
        fit_intensity=FIT_ORDER is not None,
        fit_degree=FIT_ORDER or 1,
        fit_percentiles=FIT_PERCENTILES,
        estimation_roi=ALIGNMENT_ROI,
        use_master_where_valid=True,
    )
    stitch_results.append(result)
    stitched_image_ids.append([group_id_lists[index] for index in selected])
    print(f"Polarization {polarization}; reference input {reference_index}, IDs {group_id_lists[reference_index]}")
    for item in result.prepared_frames:
        print(f"  ID {item.image_id}: shift={item.shift}, coefficients={item.coefficients}")


## Inspect alignment, normalization fits, and overlap


In [ ]:
DISPLAY_PERCENTILES = (1, 99.9)

for label, result in zip(("+", "-"), stitch_results):
    reference = result.prepared_frames[0]
    rows = len(result.prepared_frames)
    estimation_region = np.ones(reference.image.shape, dtype=bool)
    if ALIGNMENT_ROI is not None:
        estimation_region[:] = False
        estimation_region[ALIGNMENT_ROI] = True
    fig, axes = plt.subplots(rows, 4, figsize=(20, 4 * rows), squeeze=False)
    for row, item in enumerate(result.prepared_frames):
        overlap = reference.valid & item.valid
        fit_overlap = overlap & estimation_region
        difference = np.where(overlap, item.image - reference.image, np.nan)
        values = item.image[item.valid & np.isfinite(item.image)]
        vmin, vmax = np.percentile(values, DISPLAY_PERCENTILES)
        residual_limit = np.nanpercentile(np.abs(difference), 99) if overlap.any() else 1
        axes[row, 0].imshow(item.image, cmap="viridis", vmin=vmin, vmax=vmax)
        axes[row, 0].set_title(f"{label} ID {item.image_id}: aligned and calibrated")
        axes[row, 1].imshow(item.valid, cmap="gray", vmin=0, vmax=1)
        axes[row, 1].set_title("valid pixels after mask and alignment")
        axes[row, 2].imshow(difference, cmap="coolwarm", vmin=-residual_limit, vmax=residual_limit)
        axes[row, 2].set_title("difference from reference in valid overlap")

        # Scatter points and the polynomial used to map this image to the master.
        moving_values = item.fit_input[fit_overlap]
        reference_values = reference.image[fit_overlap]
        low, high = np.percentile(moving_values, FIT_PERCENTILES)
        fitted = (moving_values >= low) & (moving_values <= high)
        plot_step = max(1, fitted.sum() // 5000)
        axes[row, 3].scatter(
            moving_values[fitted][::plot_step], reference_values[fitted][::plot_step],
            s=3, alpha=0.2, color="tab:blue", label="fit pixels",
        )
        fit_x = np.linspace(low, high, 300)
        axes[row, 3].plot(
            fit_x, np.polyval(item.coefficients, fit_x),
            color="red", linewidth=2, label=f"order {len(item.coefficients) - 1} fit",
        )
        axes[row, 3].set_title("master intensity vs input intensity")
        axes[row, 3].set_xlabel("input intensity")
        axes[row, 3].set_ylabel("master intensity")
        axes[row, 3].legend()
        for axis in axes[row, :3]:
            axis.set_axis_off()
    plt.tight_layout()
    plt.show()

## Inspect the final combined images


In [ ]:
for polarization, image_ids, result in zip(("+", "-"), stitched_image_ids, stitch_results):
    image_values = result.image[result.missing_mask == 0]
    vmin, vmax = np.percentile(image_values, (1, 99.9))
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    shown = axes[0].imshow(result.image, cmap="viridis", vmin=vmin, vmax=vmax)
    axes[0].set_title(f"{polarization}: final stitched image")
    fig.colorbar(shown, ax=axes[0], label="Normalized intensity")
    axes[1].imshow(result.source_count, cmap="viridis")
    axes[1].set_title("number of contributing images")
    axes[2].imshow(result.missing_mask, cmap="gray", vmin=0, vmax=1)
    axes[2].set_title("final mask_pixel")
    for axis in axes:
        axis.set_axis_off()
    fig.suptitle(f"Reference IDs {image_ids[0]}; input groups {image_ids}")
    plt.tight_layout()
    plt.show()


## Simple names for interactive inspection

The names below are intentionally stable. Use them directly in later cells; no knowledge of the stitching loops is required.


In [ ]:
pos_reference = np.asarray(frames[PLUS_REFERENCE_INPUT].image, dtype=float)
neg_reference = np.asarray(frames[MINUS_REFERENCE_INPUT].image, dtype=float)
mask_pos_reference = np.asarray(mask_pixels[PLUS_REFERENCE_INPUT], dtype=np.uint8)
mask_neg_reference = np.asarray(mask_pixels[MINUS_REFERENCE_INPUT], dtype=np.uint8)

pos = np.asarray(stitch_results[0].image, dtype=float)
neg = np.asarray(stitch_results[1].image, dtype=float)
mask_pos = np.asarray(stitch_results[0].missing_mask, dtype=np.uint8)
mask_neg = np.asarray(stitch_results[1].missing_mask, dtype=np.uint8)


def show_image(image, title="image", percentiles=(1, 99.9)):
    """Display any 2-D NumPy image with a useful linear intensity range."""
    image = np.asarray(image, dtype=float)
    finite = image[np.isfinite(image)]
    vmin, vmax = np.percentile(finite, percentiles)
    fig, axis = plt.subplots(figsize=(6, 5))
    shown = axis.imshow(image, cmap="viridis", vmin=vmin, vmax=vmax)
    axis.set_title(title)
    axis.set_axis_off()
    fig.colorbar(shown, ax=axis, label="Intensity")
    plt.tight_layout()
    plt.show()
    return fig, axis


def show_input(input_number):
    """Display one averaged, dark-corrected stitching input and its mask."""
    show_image(frames[input_number].image, f"input {input_number}: IDs {group_id_lists[input_number]}")


print("Easy NumPy names: pos_reference, neg_reference, pos, neg")
print("Easy mask names: mask_pos_reference, mask_neg_reference, mask_pos, mask_neg")
print("Examples: show_input(2) or show_image(pos, 'stitched positive')")


## Save files for 01_FTH.ipynb

In [ ]:
OUTPUT_FOLDER = BASEFOLDER / "processed" / "stitched"
OUTPUT_NAME = "moved_beamstop"
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

output_files = []
for polarization, image_ids, result in zip(("+", "-"), stitched_image_ids, stitch_results):
    name = "plus" if polarization == "+" else "minus"
    output_file = OUTPUT_FOLDER / f"stitched_{OUTPUT_NAME}_{name}_{USER}.npz"
    fit_coefficients = np.asarray([item.coefficients for item in result.prepared_frames], dtype=float)
    image_shifts = np.asarray([item.shift for item in result.prepared_frames], dtype=float)
    factors = fit_coefficients[:, 0] if FIT_ORDER == 1 else np.full(len(image_ids), np.nan)
    offsets = fit_coefficients[:, -1]

    # Detector defects are fixed. The remaining missing region is the final beamstop mask.
    mask_detector = np.asarray(mask_detectors[0], dtype=np.uint8)
    mask_beamstop = (
        (result.missing_mask > 0) & (mask_detector == 0)
    ).astype(np.uint8)
    mask_pixel = np.clip(mask_detector + mask_beamstop, 0, 1).astype(np.uint8)

    reference_ids = image_ids[0]
    reference_input = PLUS_REFERENCE_INPUT if polarization == "+" else MINUS_REFERENCE_INPUT
    reference_exposure = frames[reference_input].exposure
    np.savez_compressed(
        output_file,
        image=result.image,
        mask_detector=mask_detector,
        mask_beamstop=mask_beamstop,
        mask_pixel=mask_pixel,
        source_count=result.source_count,
        reference_exposure=float(reference_exposure),
        reference_image_id=int(reference_ids[0]),
        reference_image_ids=np.asarray(reference_ids, dtype=int),
        ordered_input_groups=np.asarray(["+".join(map(str, ids)) for ids in image_ids]),
        polarization=np.asarray(polarization),
        image_shifts=image_shifts,
        fit_order=-1 if FIT_ORDER is None else FIT_ORDER,
        fit_coefficients=fit_coefficients,
        factors=factors,
        offsets=offsets,
        energy_eV=energy_eV,
    )
    output_files.append(output_file)
    print(f"Saved {polarization}: {output_file}")

print("Use these as PLUS_STITCHED_FILE and MINUS_STITCHED_FILE in 01_FTH:")
for polarization, output_file in zip(("+", "-"), output_files):
    print(f"  {polarization}: {output_file.relative_to(BASEFOLDER)}")


In [ ]:
print("image ID groups:", IMAGE_ID_GROUPS)
print("dark ID groups:", DARK_ID_GROUPS if INPUT_KIND == "raw" else "already subtracted")
print("plus reference input:", PLUS_REFERENCE_INPUT, group_id_lists[PLUS_REFERENCE_INPUT])
print("minus reference input:", MINUS_REFERENCE_INPUT, group_id_lists[MINUS_REFERENCE_INPUT])
